In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

from catboost import CatBoostClassifier, Pool
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

print("\n=== CARREGAR DADOS ===")
train = pd.read_csv("datasets/training_data.csv", encoding="latin1")
test  = pd.read_csv("datasets/test_data.csv", encoding="latin1")
example = pd.read_csv("datasets/example_submission.csv", encoding="latin1")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

# ================================
# TARGET
# ================================
print("\n=== PREPARAR TARGET ===")
y = train["AVERAGE_SPEED_DIFF"].fillna("None")
X = train.drop("AVERAGE_SPEED_DIFF", axis=1)

# ================================
# FEATURE ENGINEERING DO record_date
# ================================
def process_dates(df):
    df["record_date"] = pd.to_datetime(df["record_date"])
    df["year"] = df["record_date"].dt.year
    df["month"] = df["record_date"].dt.month
    df["day"] = df["record_date"].dt.day
    df["hour"] = df["record_date"].dt.hour
    df["weekday"] = df["record_date"].dt.weekday
    return df.drop("record_date", axis=1)

X = process_dates(X)
test = process_dates(test)

# ================================
# COLUNAS
# ================================
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(include=["number"]).columns.tolist()

print("\nCategóricas:", cat_cols)
print("Numéricas:", num_cols)

# ================================
# TRATAR NAN
# ================================
print("\n=== TRATAR NAN ===")
for col in cat_cols:
    X[col] = X[col].fillna("Unknown").astype(str)
for col in num_cols:
    X[col] = X[col].fillna(X[col].median())

# ================================
# SPLIT
# ================================
print("\n=== CRIAR VALIDATION SPLIT ===")
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# ================================
# ONE-HOT PARA LGBM E RF
# ================================
print("\n=== ONE-HOT ENCODING PARA LGBM / RF ===")

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

# FIT transform no treino
Xl_train = ohe.fit_transform(X_train[cat_cols])
Xl_val   = ohe.transform(X_val[cat_cols])

# juntar as numéricas
Xl_train = np.hstack([Xl_train, X_train[num_cols].values])
Xl_val   = np.hstack([Xl_val,   X_val[num_cols].values])

# Encode target
target_enc = LabelEncoder()
y_train_enc = target_enc.fit_transform(y_train)
y_val_enc   = target_enc.transform(y_val)

# ================================
# CATBOOST
# ================================
print("\n=== TREINAR CATBOOST ===")

train_pool = Pool(X_train, y_train, cat_features=cat_cols)
val_pool   = Pool(X_val, cat_features=cat_cols)

cat_model = CatBoostClassifier(
    iterations=700,
    depth=8,
    learning_rate=0.04,
    loss_function="MultiClass",
    verbose=100
)

cat_model.fit(train_pool, eval_set=val_pool, verbose=100)

cat_val_preds = cat_model.predict(val_pool).flatten()
print("\nCATBOOST ACC:", accuracy_score(y_val, cat_val_preds))
print(classification_report(y_val, cat_val_preds))

# ================================
# LIGHTGBM
# ================================
print("\n=== TREINAR LIGHTGBM ===")

lgbm = LGBMClassifier(
    n_estimators=800,
    learning_rate=0.04,
    num_leaves=60,
    objective="multiclass",
    num_class=len(target_enc.classes_)
)

lgbm.fit(Xl_train, y_train_enc)
lgbm_val_preds = target_enc.inverse_transform(lgbm.predict(Xl_val))

print("\nLIGHTGBM ACC:", accuracy_score(y_val, lgbm_val_preds))
print(classification_report(y_val, lgbm_val_preds))

# ================================
# RANDOM FOREST
# ================================
print("\n=== TREINAR RF ===")

rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=18,
    n_jobs=-1
)

rf.fit(Xl_train, y_train_enc)
rf_val_preds = target_enc.inverse_transform(rf.predict(Xl_val))

print("\nRF ACC:", accuracy_score(y_val, rf_val_preds))
print(classification_report(y_val, rf_val_preds))

# ================================
# STACKING
# ================================
print("\n=== STACKING ===")

cat_proba = cat_model.predict_proba(val_pool)
lgbm_proba = lgbm.predict_proba(Xl_val)
rf_proba = rf.predict_proba(Xl_val)

stack_input = np.hstack([cat_proba, lgbm_proba, rf_proba])

meta = LogisticRegression(max_iter=1000)
meta.fit(stack_input, y_val)

stack_preds = meta.predict(stack_input)

print("\nSTACKING ACC:", accuracy_score(y_val, stack_preds))
print(classification_report(y_val, stack_preds))

# ================================
# GERAR SUBMISSÃO
# ================================
print("\n=== SUBMISSION ===")

# preprocess test
Xc_test = test.copy()

# One-hot do test
Xl_test = ohe.transform(Xc_test[cat_cols])
Xl_test = np.hstack([Xl_test, Xc_test[num_cols].values])

# CatBoost
# --- FIX FINAL: CatBoost não aceita NaN ---
for col in cat_cols:
    Xc_test[col] = Xc_test[col].astype(str).fillna("Unknown")

for col in num_cols:
    Xc_test[col] = Xc_test[col].fillna(Xc_test[col].median())

test_pool = Pool(Xc_test, cat_features=cat_cols)

cat_p = cat_model.predict_proba(test_pool)
lgb_p = lgbm.predict_proba(Xl_test)
rf_p  = rf.predict_proba(Xl_test)

stack_final = np.hstack([cat_p, lgb_p, rf_p])
final_preds = meta.predict(stack_final)

submission = pd.DataFrame({
    "RowId": example["RowId"],
    "Speed_Diff": final_preds
})

submission.to_csv("submission.csv", index=False)
print("\n✔ submission.csv criado!")

print("\n=== FIM ===")


=== CARREGAR DADOS ===
Train shape: (6812, 14)
Test shape: (1500, 13)

=== PREPARAR TARGET ===

Categóricas: ['city_name', 'LUMINOSITY', 'AVERAGE_CLOUDINESS', 'AVERAGE_RAIN']
Numéricas: ['AVERAGE_FREE_FLOW_SPEED', 'AVERAGE_TIME_DIFF', 'AVERAGE_FREE_FLOW_TIME', 'AVERAGE_TEMPERATURE', 'AVERAGE_ATMOSP_PRESSURE', 'AVERAGE_HUMIDITY', 'AVERAGE_WIND_SPEED', 'AVERAGE_PRECIPITATION', 'year', 'month', 'day', 'hour', 'weekday']

=== TRATAR NAN ===

=== CRIAR VALIDATION SPLIT ===

=== ONE-HOT ENCODING PARA LGBM / RF ===

=== TREINAR CATBOOST ===
0:	learn: 1.5302007	total: 84.8ms	remaining: 59.2s
